In [ ]:
from lib.opt_types import *
from lib.utils import *
from lib.part_one import f, x_zero

# Stochastic gradient methods - 35 Points


In this part, you will implement stochastic gradient methods and compare their convergence on the logistic regression problem using the 3,376-shot Golden State Warriors subset from Part 2. You will then compare uniform and importance sampling on a separate test problem.

We write the regularized logistic regression objective introduced in [Part 2, equation (3)](Part%202%20-%20First%20order%20methods.ipynb#eq-binary-objective) in finite-sum form:

<a id="eq-objective"></a>

$$
\tag{6}
f(\mathbf{x}) = \frac{1}{n}\sum_{i=1}^n\bigg\{\underbrace{\log(1 + \exp(- b_i \mathbf{a}_i^T\mathbf{x})) + \frac{\mu}{2}\|\mathbf{x}\|^2}_{f_i(\mathbf{x})}\bigg\},
$$

We suppress the dependence on $\mu$ in the notation. The component gradients are

$$
\nabla f_i(\mathbf{x}) = -b_i \sigma(-b_i \cdot \mathbf{a}_i^T\mathbf{x})\mathbf{a}_i + \mu \mathbf{x}.
$$

The `Function` type provides access to the individual terms through the following attributes:

- To access the gradient of the `j`-th term at a point `x` you can write `f.i_grad(j, x)`.
- The number of terms $n$ is stored in the attribute `f.n`. 

Consider the following stochastic gradient update: At the iteration $k$, pick $i_k\in\{1, \ldots, n\}$ uniformly at random and define

<a id="eq-sgd"></a>

$$
\mathbf{x}^{k+1} :=  \mathbf{x}^k - \alpha_k\nabla{f}_{i_k}(\mathbf{x}^k).\tag{7}
$$

__(a)__ (2 points) Show that $\nabla f_{i_k}(\mathbf{x})$ is an unbiased estimator of $\nabla f(\mathbf{x})$. Explain why $\nabla f_{i_k}$ is Lipschitz continuous with the bound $L(f_{i_k}) = \|\mathbf{a}_{i_k}\|^2+ \mu $. 

__Hint__: Recall how we computed $L$ in Problem 1. In the following, we will set $L_{\max} = \max_{i\in\{1, \ldots, n\}}L(f_i)$.

*Write your answer here.*


__(b)__ (2 points) 
Implement the [SGD update (7)](#eq-sgd) for [objective (6)](#eq-objective) by completing the following cells with $\alpha_k =\frac{0.01}{k}$, starting the counter at $k=1$.

**Hint**: For some `N`, the function `np.random.choice(N)` generates a random integer in $\{0, \dots, N-1 \}$.


In [ ]:
@dataclass
class SGD_state(OptState):
    x_k: Vector
    k: int

In [ ]:
def SGD_update(f, state):
    x_k, k = state
    i_k = ...  # FILL
    next_x_k = ...  # FILL
    return SGD_state(next_x_k, k+1)


def SGD_initialize(f, x_zero):
    return SGD_state(x_zero, 1)


In [ ]:
SGD = OptAlgorithm(name="SGD", init_state=SGD_initialize, state_update=SGD_update)

__(c)__ (6 points) 
The stochastic average gradient method (`SAG`) stores the most recently computed gradient of each component and uses their average in the update:

<a id="eq-sag"></a>

$$
\tag{8}
\begin{cases}
\text{pick } i_k\in\{1, \ldots, n\} \text{ uniformly at random}\\
\mathbf{x}^{k+1} := \mathbf{x}^k - \frac{\alpha_k}{n}\sum_{i=1}^n\mathbf{v}_i^k,
\end{cases}
$$

where 

$$
\mathbf{v}_i^k =
\begin{cases}
\nabla f_i(\mathbf{x}^k) &\text{if}\, i = i_k,\\
\mathbf{v}_i^{k-1} &\text{otherwise}.
\end{cases}
$$

Implement the [SAG update (8)](#eq-sag) for [objective (6)](#eq-objective) by completing the following cells with $\alpha_k=\frac{0.01}{L_{\max}}$, initializing all stored gradients to zero. You can access $L_{\max}$ through `f.L_max`.

In [ ]:
@dataclass
class SAG_state(OptState):
    x_k: Vector
    v_k: List[Vector]
    alpha_k: float

In [ ]:
def SAG_update(f, state):
    # FILL
    raise NotImplementedError("Implement SAG_update.")

def SAG_initialize(f, x_zero):
    # FILL
    raise NotImplementedError("Implement SAG_initialize.")


In [ ]:
SAG = OptAlgorithm(name="SAG", init_state=SAG_initialize, state_update=SAG_update)

In [ ]:
plot_epochs([SGD, SAG], f, x_zero, 500000, x_axis="gradient_evaluations")

__(d)__ (6 points) 
We can improve the convergence rate of SGD by periodically computing the full gradient. `SVRG` uses the following variance reduction scheme:

$$
\begin{cases}
\text{if } k = 0 \text{ mod } q:\\
\quad \mathbf{z} = \mathbf{x}^k\\
\quad \tilde{\mathbf{v}} = \nabla f(\mathbf{x}^k)\\
\text{Pick } i_k\in \{1, \ldots, n\} \text{ uniformly at random}\\
\mathbf{d}^k = \nabla f_{i_k}({\mathbf{x}}^k) - \nabla f_{i_k}(\mathbf{z}) + \tilde{\mathbf{v}}\\
{\mathbf{x}}^{k+1} := {\mathbf{x}}^k - \gamma\mathbf{d}^k\\
\end{cases}
$$

Implement `SVRG` by completing the following cells with $\gamma = 1/L_{\max}$ and $q = 100$.

In [ ]:
@dataclass
class SVRG_state(OptState):
    x_k: Vector 
    z: Vector
    tilde_v: Vector
    q: int
    gamma: float
    k: int

In [ ]:
def SVRG_update(f, state):
    # FILL
    raise NotImplementedError("Implement SVRG_update.")

def SVRG_initialize(f, x_zero):
    # FILL
    raise NotImplementedError("Implement SVRG_initialize.")


In [ ]:
SVRG = OptAlgorithm(name="SVRG", init_state=SVRG_initialize, state_update=SVRG_update)

__(e)__ (6 points) Another variance reduction method is `Spider`. The scheme can be described as follows:

$$
\begin{cases}
\text{if } k = 0 \text{ mod } q:\\
\quad {\mathbf{v}}^k = \nabla f({\mathbf{x}}^k)\\
\text{if } k \neq 0 \text{ mod } q:\\
\quad \text{Pick } i_k\in \{1, \ldots, n\} \text{ uniformly at random}\\
\quad {\mathbf{v}}^k = \nabla f_{i_k}({\mathbf{x}}^k) - \nabla f_{i_k}({\mathbf{x}}^{k-1}) + {\mathbf{v}}^{k-1}\\
\eta := \min \big(\frac{1}{\left \| {\mathbf{v}}^{k} \right \|_2}, \frac{1}{\epsilon}\big)\\
{\mathbf{x}}^{k+1} := {\mathbf{x}}^k - \gamma \eta {\mathbf{v}}^{k}\\
\end{cases}
$$


Implement `Spider` by completing the following cells. Pick `q=100`, $\epsilon = 0.05$ and $\gamma = \frac{1}{L_\text{max}}$.

In [ ]:
@dataclass
class Spider_state(OptState):
    x_k: Vector 
    prev_x_k: Vector
    prev_v_k: Vector
    q: int
    gamma: float
    k: int
    epsilon: float

In [ ]:
import numpy
from numpy.linalg import norm

def Spider_update(f, state):
    # FILL
    raise NotImplementedError("Implement Spider_update.")

def Spider_initialize(f, x_zero):
    # FILL
    raise NotImplementedError("Implement Spider_initialize.")


In [ ]:
Spider = OptAlgorithm(name="Spider", init_state=Spider_initialize, state_update=Spider_update)

__(f)__ (6 points) Another variance reduction method is `STORM` (STOchastic Recursive Momentum), introduced by [Cutkosky and Orabona](https://arxiv.org/abs/1905.10018). We consider the following zero-initialized version.

Sample $i_0$ uniformly from $\{1,\ldots,n\}$ and initialize

$$
\mathbf{v}^0 = \mathbf{0},
\qquad S_0 = \|\nabla f_{i_0}(\mathbf{x}^0)\|_2^2.
$$

The scheme can be described as follows:

$$
\begin{cases}
\eta_k = \frac{\kappa}{(w + S_k)^{1/3}},\\
\mathbf{x}^{k+1} = \mathbf{x}^k - \eta_k\mathbf{v}^k,\\
a_{k+1} = c\eta_k^2,\\
\text{Pick } i_{k+1}\in\{1,\ldots,n\}\text{ uniformly at random},\\
\mathbf{g}^{k+1} = \nabla f_{i_{k+1}}(\mathbf{x}^{k+1}),\\
\mathbf{v}^{k+1} = \mathbf{g}^{k+1}
  + (1-a_{k+1})\big(\mathbf{v}^k - \nabla f_{i_{k+1}}(\mathbf{x}^k)\big),\\
S_{k+1} = S_k + \|\mathbf{g}^{k+1}\|_2^2.
\end{cases}
$$

Implement `STORM` by completing the following cells. Pick

$$
\eta_\text{max} = \frac{15}{L_\text{max}},\qquad
w = 0.1L_\text{max},\qquad
\kappa = \eta_\text{max}w^{1/3},\qquad
c = \frac{0.03}{\eta_\text{max}^2}.
$$


In [ ]:
@dataclass
class Storm_state(OptState):
    x_k: Vector
    v_k: Vector
    sum_grad_sq: float
    kappa: float
    w: float
    c: float


In [ ]:
def Storm_update(f, state):
    # FILL
    raise NotImplementedError("Implement Storm_update.")


def Storm_initialize(f, x_zero):
    # FILL
    raise NotImplementedError("Implement Storm_initialize.")


In [ ]:
Storm = OptAlgorithm(name="STORM", init_state=Storm_initialize, state_update=Storm_update)

In [ ]:
np.random.seed(0)
plot_epochs([SVRG, Spider, Storm], f, x_zero, 50000,
            x_axis="epochs")


__(g)__ (2 points) In the above plot, the number of epochs is defined as the number of individual sample gradient evaluations divided by the number of data samples $n$. Although SVRG, Spider, and STORM each perform 50,000 updates, their curves cover different ranges on the horizontal axis. Explain why.


*Write your answer here.*


__(h)__ (5 points) **Uniform and importance sampling.**

The SGD method studied so far uses **uniform sampling (SGD-US)**: for a finite-sum objective $f=\frac{1}{n}\sum_{i=1}^n f_i$, it selects every component with equal probability. However, the components can have very different gradient Lipschitz constants $L_i$. Sampling components with larger $L_i$ more often can accelerate convergence. **SGD with importance sampling (SGD-IS)** implements this idea by choosing each component with probability proportional to $L_i$ and reweighting its gradient to keep the estimate unbiased. For background, see [Gower et al. (2019)](https://arxiv.org/pdf/1901.09401) and [Zhao and Zhang (2015)](https://proceedings.mlr.press/v37/zhaoa15.pdf).

Both SGD-US and SGD-IS are special cases of the general **SGD with non-uniform sampling (SGD-NS)** scheme, which allows arbitrary positive sampling probabilities, including uniform probabilities.

SGD-NS is:

Initialize $\mathbf{x}^0$, a step size $\gamma>0$, and probabilities $p_i>0$ summing to one. For $k=0,1,\ldots$:

$$
\begin{cases}
\text{Pick } i_k\in\{1,\ldots,n\}\text{ with probability }p_{i_k},\\
\mathbf{g}^k = \dfrac{\nabla f_{i_k}(\mathbf{x}^k)}{n p_{i_k}},\\
\mathbf{x}^{k+1} = \mathbf{x}^k - \gamma\mathbf{g}^k.
\end{cases}
$$

Then SGD-US and SGD-IS use the following choices of sampling probabilities and step sizes in this implementation:

| Method | Sampling probabilities | Step size |
|---|---|---|
| SGD-US | $p_i=1/n$ | $\gamma=1/\max_i L_i$ |
| SGD-IS | $p_i=L_i/\sum_j L_j$ | $\gamma=1/(\frac{1}{n}\sum_i L_i)$ |

<small>Here we use constant learning rates in both SGD-US and SGD-IS, rather than the diminishing learning rates commonly used in the standard theoretical analysis of SGD. This is a problem-specific choice intended to simplify the implementation, rather than a general recommendation for SGD.</small>

For this comparison, the helper supplies a separate test objective, passed to each algorithm as `f`, with the following interface:

- `f.n`: the number of component functions $f_i$.
- `f.i_grad(i, x)`: the gradient $\nabla f_i(x)$ of component `i` at `x`.
- `f[i].L`: the Lipschitz constant $L_i$ of the gradient $\nabla f_i$.

To sample a component index according to the probabilities specified above, use `np.random.choice(f.n, p=p)`, where `p` is a list or one-dimensional array containing one probability for each component, with entries summing to one. The call returns index `i` with probability `p[i]`.

Please implement SGD-US and SGD-IS by completing the initialization and update functions in the following cells, then run the final cell to compare their convergence.

In [ ]:
from lib.quadratic_sampling import make_quadratic_problem, plot_sampling

f_quadratic, x_quadratic = make_quadratic_problem()

In [ ]:
@dataclass
class SGD_US_state(OptState):
    x_k: Vector
    gamma: float

In [ ]:
def SGD_US_update(f, state):
    # FILL
    raise NotImplementedError("Implement SGD_US_update.")


def SGD_US_initialize(f, x_zero):
    # FILL
    raise NotImplementedError("Implement SGD_US_initialize.")


In [ ]:
SGD_US = OptAlgorithm(name="SGD-US", init_state=SGD_US_initialize, state_update=SGD_US_update)

In [ ]:
@dataclass
class SGD_IS_state(OptState):
    x_k: Vector
    p: Vector
    gamma: float

In [ ]:
def SGD_IS_update(f, state):
    # FILL
    raise NotImplementedError("Implement SGD_IS_update.")


def SGD_IS_initialize(f, x_zero):
    # FILL
    raise NotImplementedError("Implement SGD_IS_initialize.")


In [ ]:
SGD_IS = OptAlgorithm(name="SGD-IS", init_state=SGD_IS_initialize, state_update=SGD_IS_update)

In [ ]:
plot_sampling([SGD_US, SGD_IS], f_quadratic, x_quadratic)